## Private repository authentication

Before running cell 1, open Colab's **Secrets (key icon)** panel, create `GITHUB_TOKEN` using a fine-grained GitHub personal access token with **Contents: read-only** access to `Ko4rut/ML-Advance-Group`, and allow this notebook to access the secret. Do not put tokens into notebook cells, Git remote URLs, or outputs.

# Xiangqi — CPU Batch Game Collector on Colab

Chạy từ **nhánh `feature/game-data-pipeline`**, không dùng FastAPI/frontend. Thu thập record theo dạng JSONL (1 ván / 1 dòng).

**Lưu ý sử dụng:** Colab Free liệt kê *chess training* trong danh sách bị hạn chế. Sử dụng Colab trả phí với compute units phù hợp hoặc máy chủ CPU; không dùng notebook này để né giới hạn dịch vụ. [Colab FAQ](https://research.google.com/colaboratory/faq.html).

**Runtime → Change runtime type → Hardware accelerator: None (CPU).** Alpha-Beta Python hiện không tận dụng GPU.

## 1. Kiểm tra tài nguyên và tải source

In [ ]:
import os
import platform
import subprocess
import sys
import tempfile
from pathlib import Path

from google.colab import userdata

print("Python:", sys.version.split()[0])
print("CPU vCPUs:", os.cpu_count())
print("Architecture:", platform.platform())

# This is a private GitHub repository. Add a Colab Secret named GITHUB_TOKEN
# with a fine-grained GitHub PAT, repository selection: ML-Advance-Group,
# repository permission: Contents (read-only), then enable notebook access.
REPO_URL = "https://github.com/Ko4rut/ML-Advance-Group.git"
REPO_DIR = Path("/content/ML-Advance-Group")
BRANCH = "feature/game-data-pipeline"

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Missing Colab Secret GITHUB_TOKEN; add it in the Secrets sidebar.")

# Git prompts for credentials through an ephemeral helper. Unlike putting
# the token inside a git URL, this does not save it in .git/config or print it.
askpass_dir = Path(tempfile.mkdtemp(prefix="xiangqi_git_", dir="/tmp"))
askpass = askpass_dir / "askpass.sh"
askpass.write_text(
    "#!/bin/sh\n"
    "case \"$1\" in\n"
    "  *Username*) printf '%s\\n' 'x-access-token' ;;\n"
    "  *Password*) printf '%s\\n' \"$COLAB_GITHUB_TOKEN\" ;;\n"
    "  *) printf '%s\\n' '' ;;\n"
    "esac\n",
    encoding="utf-8",
)
askpass.chmod(0o700)

git_env = os.environ.copy()
git_env.update({
    "GIT_ASKPASS": str(askpass),
    "GIT_TERMINAL_PROMPT": "0",
    "COLAB_GITHUB_TOKEN": token,
})
del token

try:
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
            env=git_env,
            check=True,
        )
    elif (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH],
            env=git_env,
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "switch", BRANCH],
            env=git_env,
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH],
            env=git_env,
            check=True,
        )
    else:
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository. "
            "Rename/remove that incomplete folder and run this cell again."
        )
finally:
    git_env.pop("COLAB_GITHUB_TOKEN", None)
    askpass.unlink(missing_ok=True)
    askpass_dir.rmdir()

BACKEND_DIR = REPO_DIR / "project" / "backend"
assert (BACKEND_DIR / "src" / "training" / "batch_games.py").exists()
print("Backend ready:", BACKEND_DIR)


## 2. Kiểm thử collector

Chạy unit test trước khi sinh nhiều ván. Nếu thất bại, không nên chạy batch lớn.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'unittest', 'discover',
    '-s', 'tests', '-p', 'test_batch_games.py', '-v'
], cwd=str(BACKEND_DIR), check=True)

## 3. Mount Google Drive để tránh mất record khi runtime ngắt

Collector ghi và flush **sau mỗi ván** vào file JSONL. Khi lưu trực tiếp trên Drive, dữ liệu các ván đã xong có thể được giữ lại; ván đang chạy dở không được lưu. Bạn cần cho phép mount Drive trong notebook.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = Path('/content/drive/MyDrive/XiangqiAI/datasets')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('Output folder:', SAVE_DIR)

## 4. Cấu hình thử nghiệm

Mặc định chạy **3 ván giới hạn 100 ply** để đo tốc độ, không phải để tạo tập train hoàn chỉnh. Đổi `GAMES`, `MAX_PLIES`, `OPENING_PLIES` sau khi benchmark. Nếu chưa có kết quả kết thúc hợp lệ, record sẽ có `status=truncated`, không phải draw.

In [ ]:
from datetime import datetime, timezone

RED_AGENT = 'alphabeta'
BLACK_AGENT = 'alphabeta'
RED_DEPTH = 2
BLACK_DEPTH = 2
GAMES = 3
MAX_PLIES = 100
OPENING_PLIES = 2
NO_PROGRESS_LIMIT = 0   # 0 = không dừng sớm ngoài max plies / luật engine
SEED = 42

STAMP = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
OUTPUT_FILE = SAVE_DIR / f'{RED_AGENT}_vs_{BLACK_AGENT}_{STAMP}.jsonl'
print('Output:', OUTPUT_FILE)

## 5. Sinh ván cờ, không render bàn cờ

Đếm số ván hoàn tất trên terminal của Colab. Record được ghi theo từng ván; nếu phiên chạy bị dừng, bạn có thể dùng file đã lưu làm dataset partial mà không cần chạy lại các ván đã có.

In [ ]:
command = [
    sys.executable, '-u', '-m', 'src', 'batch',
    '--red', RED_AGENT, '--black', BLACK_AGENT,
    '--red-depth', str(RED_DEPTH), '--black-depth', str(BLACK_DEPTH),
    '--games', str(GAMES), '--max-plies', str(MAX_PLIES),
    '--opening-plies', str(OPENING_PLIES),
    '--no-progress-limit', str(NO_PROGRESS_LIMIT),
    '--seed', str(SEED), '--output', str(OUTPUT_FILE),
    '--progress-every', '1'
]

print('Starting CPU batch generation...')
subprocess.run(command, cwd=str(BACKEND_DIR), check=True)

## 6. Xem tỷ lệ hòa, ván cắt ngắn và tốc độ thực tế

Nếu nhiều ván `repetition` hoặc `truncated`, chuyển lên máy mạnh hơn **không làm tăng chất lượng nhãn thắng/thua**. Hãy đa dạng hóa đối thủ và khai cuộc trước khi tăng số lượng.

In [ ]:
import collections
import json
import statistics

records = [json.loads(line) for line in OUTPUT_FILE.read_text(encoding='utf-8').splitlines() if line.strip()]
results = collections.Counter(
    (r['outcome']['status'], r['outcome']['reason']) for r in records
)
print('Records saved:', len(records))
print('Result breakdown:', dict(results))
if records:
    print('Average ply:', round(statistics.mean(r['ply_count'] for r in records), 2))
    print('Average seconds/game:', round(statistics.mean(r['elapsed_seconds'] for r in records), 2))
    print('Unique fingerprints:', len({r['fingerprint'] for r in records}))
    print('Full results:', [(r['game_index'], r['outcome']) for r in records])